In [57]:
# ===== 0) Setup =====
from __future__ import annotations

import re
import json
import math
from statistics import mean, median, stdev
from typing import List, Dict, Any, Optional
from collections import Counter

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 200)


In [28]:
TRAIN = "data/train.csv"
VAL = "data/phase_1_test.csv"
TEST = "data/phase_2_test.csv"


train_df = pd.read_csv(TRAIN)
val_df = pd.read_csv(VAL)
test_df = pd.read_csv(TEST)

val_ans = pd.read_csv("data/phase_1_test_truth - Sheet1.csv")
val_df = val_df.merge(val_ans, on="ID", how="left")

In [29]:
def sanitize_question_text(q: str) -> str:
    """
    Convert literal '\\n' to real newlines.
    CRITICAL: Don't use unicode decoding or you corrupt '\\boxed'
    """
    if q is None:
        return ""
    return str(q).replace("\\n", "\n").replace("\r\n", "\n").replace("\r", "\n")


In [ ]:
import re
import pandas as pd

# Matches \boxed{{something}}  -> \boxed{something}
BOXED_DOUBLE_BRACE_CONTENT_RE = re.compile(r"\\boxed\{\{([^{}]+)\}\}")

# Matches \boxed{{}} (or with spaces) -> \boxed{}
BOXED_DOUBLE_BRACE_EMPTY_RE = re.compile(r"\\boxed\{\{\s*\}\}")

def sanitize_question_text(q: str) -> str:
    if q is None:
        return ""
    return str(q).replace("\\n", "\n").replace("\r\n", "\n").replace("\r", "\n")

def normalize_boxed_in_text(text: str) -> str:
    """
    - \boxed{{}}      -> \boxed{ans}
    - \boxed{{ans}}   -> \boxed{ans}
    """
    if text is None:
        return ""
    t = str(text)
    t = BOXED_DOUBLE_BRACE_EMPTY_RE.sub(r"\\boxed{ans}", t)
    t = BOXED_DOUBLE_BRACE_CONTENT_RE.sub(r"\\boxed{\1}", t)
    return t

def clean_question_series(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .apply(sanitize_question_text)
         .apply(normalize_boxed_in_text)
         .str.strip()
    )

# Apply
train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

train_df["question"] = clean_question_series(train_df["question"])
val_df["question"]   = clean_question_series(val_df["question"])
test_df["question"]  = clean_question_series(test_df["question"])

# Sanity check: should be 0 now
def count_double_brace(df, name):
    n = df["question"].str.contains(r"\\boxed\{\{", regex=True, na=False).sum()
    print(f"{name}: remaining '\\\\boxed{{{{...}}}}' patterns = {int(n)}")

count_double_brace(train_df, "train_df")
count_double_brace(val_df, "val_df")
count_double_brace(test_df, "test_df")


train_df: remaining '\\boxed{{...}}' patterns = 0
val_df: remaining '\\boxed{{...}}' patterns = 0
test_df: remaining '\\boxed{{...}}' patterns = 0


In [31]:
for i in range(3):
    print("Q:", train_df.loc[i, "question"])
    print("A:", train_df.loc[i, "answer"])
    print("---")

Q: Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{ans} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user throughput.
C8: Average scheduled RBs are below 160, affecting throughput.

Given:
- The default electronic downtilt value is 255, representing a downtilt angle of 6 degrees. Other values represent th

*1. load my dataset which contains the original ID, question, and answer. and clean the question tab so that all "\boxed{{}}" to have only "\boxed{ans}" *
recreate the various answer formats available in the dataset. 
Parse the tables of each questions to extract the 48 relevant features. 
create the reasoning prompt for each answer based on the relevant features 
Create the system prompt: This is very critical as we need to adjust the system prompt to account for the fact that there are 2 question types, A) 5G Root Cause Analysis (RCA) - Drive test data analysis, and B) General multiple-choice questions - History, math, logic, etc. we need to consider if we provide the different feature types and how to calculate them in the decision flow for Type A (5G Root Cause Analysis (RCA) ) so the model can recreate those features before going into the generated reasoning or keeping it simple. and then we also have to provide a strong decision flow for type B logical/reasoning questions

In [37]:
import re
import pandas as pd
from typing import List, Dict, Any, Optional
import random

# -------------------------
# A) Utilities
# -------------------------

BOXED_RE = re.compile(r"\\boxed\{([^{}]*)\}")

def extract_answer_token(ans: Any) -> Optional[str]:
    """
    Extract token inside \\boxed{...} if present, else return stripped string.
    """
    if ans is None or (isinstance(ans, float) and pd.isna(ans)):
        return None
    s = str(ans).strip()
    m = BOXED_RE.search(s)
    if m:
        tok = m.group(1).strip()
        return tok if tok else None
    return s if s else None


# option line matcher: supports "1:", "A:", "P3:", "M4:", "C7:" etc.
OPTION_LINE_RE = re.compile(r"(?m)^\s*(?P<label>(?:[A-H]|[A-Z]\d+|\d+))\s*:\s*(?P<text>.+?)\s*$")

def parse_options(question: str) -> List[Dict[str, str]]:
    q = question or ""
    opts = []
    for m in OPTION_LINE_RE.finditer(q):
        opts.append({"label": m.group("label").strip(), "text": m.group("text").strip()})
    return opts

def detect_answer_style_from_options(opts: List[Dict[str, str]]) -> str:
    if not opts:
        return "none"
    labels = [o["label"] for o in opts]
    if all(re.fullmatch(r"\d+", x) for x in labels):
        return "plain"
    if all(re.fullmatch(r"[A-H]", x) for x in labels):
        return "letter"
    if all(re.fullmatch(r"[A-Z]\d+", x) for x in labels):
        return "prefixed"
    return "mixed"


# -------------------------
# B) Question type (A_RCA vs B_GENERAL)
# -------------------------
DRIVE_PAT = re.compile(r"User\s+plane\s+drive\s+test\s+data\s+as\s+follows\s*[:：]\s*", re.IGNORECASE)
ENG_PAT   = re.compile(r"(?:Engeneering|Engineering)\s+parameters\s+data\s+as\s+follows\s*[:：]\s*", re.IGNORECASE)

def detect_question_type(question: str) -> str:
    q = question or ""
    ql = q.lower()
    if DRIVE_PAT.search(q) and ENG_PAT.search(q):
        return "A_RCA"
    keywords = ["rsrp", "sinr", "throughput", "handover", "gnodeb", "rb", "pci", "drive test", "engineering parameters"]
    return "A_RCA" if any(k in ql for k in keywords) else "B_GENERAL"


# -------------------------
# C) Canonical cause mapping for 5G RCA (by option TEXT meaning)
# -------------------------
CAUSE_KEYWORDS = {
    "C1": [r"downtilt", r"tilt angle.*too large", r"weak coverage.*far", r"far end"],
    "C2": [r"exceeds\s*1\s*km", r"over[-\s]?shoot", r"overshoot", r"coverage distance"],
    "C3": [r"neighboring cell provides higher throughput", r"higher throughput", r"better throughput", r"better neighbor", r"wrong cell"],
    "C4": [r"non[-\s]?colocated", r"overlapping coverage", r"co[-\s]?frequency", r"severe overlapping"],
    "C5": [r"frequent handovers", r"ping[-\s]?pong", r"handover.*degrad", r"back[-\s]?and[-\s]?forth"],
    "C6": [r"same pci mod\s*30", r"pci mod\s*30", r"pci collision", r"pci reuse", r"interference.*pci"],
    "C7": [r"speed exceeds\s*40", r">?\s*40\s*km/h", r"high speed", r"vehicle speed"],
    "C8": [r"scheduled rbs", r"rb.*below\s*160", r"average.*rb", r"resource congestion", r"high rb.*low tp"],
}

def canonical_cause_from_option_text(option_text: str) -> Optional[str]:
    t = (option_text or "").lower()
    for cause, pats in CAUSE_KEYWORDS.items():
        if any(re.search(p, t, flags=re.IGNORECASE) for p in pats):
            return cause
    return None

def build_cause_to_label_map(question: str) -> Dict[str, str]:
    """
    Maps canonical cause (C1..C8) -> label used in THIS question (e.g. "6", "P7", "M4")
    """
    opts = parse_options(question)
    out = {}
    for o in opts:
        cause = canonical_cause_from_option_text(o["text"])
        if cause and cause not in out:
            out[cause] = o["label"]
    return out

def infer_canonical_cause_from_answer(question: str, answer_token: str) -> Optional[str]:
    """
    For RCA: map answer_token (like '6' or 'P7') back to canonical cause via option text.
    """
    if not answer_token:
        return None
    opts = parse_options(question)
    for o in opts:
        if o["label"].strip() == answer_token.strip():
            return canonical_cause_from_option_text(o["text"])
    return None


# -------------------------
# D) Relabel options (plain / letter / P / S / I)
#    (preserve order & texts)
# -------------------------
def relabel_options_keep_order(question: str, to_style: str) -> str:
    opts = parse_options(question)
    if not opts:
        return question

    # compute new labels based on ordinal position
    def new_label(i: int) -> str:
        if to_style == "plain":
            return str(i)
        if to_style == "letter":
            return chr(64 + i)  # A..H
        # prefix style like P, S, I
        return f"{to_style}{i}"

    # build mapping old->new for each option line in order of appearance
    mapping = {}
    for i, o in enumerate(opts, start=1):
        mapping[o["label"]] = new_label(i)

    def repl(m):
        old = m.group("label").strip()
        text = m.group("text")
        return f"{mapping.get(old, old)}: {text}"

    return OPTION_LINE_RE.sub(repl, question)


# -------------------------
# E) Safe "M1..M5" generator (ONLY for subset of causes)
#    Using the same semantics as your example:
#    M1=C8 (RB<160), M2=C2 (overshoot), M3=C6 (PCI mod30), M4=C5 (frequent HO), M5=C4 (overlapping)
# -------------------------
M5_ORDER = [("M1","C8"), ("M2","C2"), ("M3","C6"), ("M4","C5"), ("M5","C4")]

def to_M5_question(question: str) -> Optional[str]:
    """
    Creates an M1..M5 version by selecting the matching option texts from the original.
    Returns None if it can't find enough matching options.
    """
    opts = parse_options(question)
    if not opts:
        return None

    # map cause -> original option text
    cause_to_text = {}
    for o in opts:
        c = canonical_cause_from_option_text(o["text"])
        if c and c not in cause_to_text:
            cause_to_text[c] = o["text"]

    # must have all required causes for M1..M5
    if not all(cause in cause_to_text for _, cause in M5_ORDER):
        return None

    # rebuild option block lines in M order
    m_lines = [f"{m}: {cause_to_text[cause]}" for m, cause in M5_ORDER]

    # replace existing option lines in the question with the new M block
    # simplest: remove all existing option lines then insert M block after "From the following..."
    q_lines = (question or "").splitlines()
    cleaned_lines = []
    for ln in q_lines:
        if OPTION_LINE_RE.match(ln):
            continue
        cleaned_lines.append(ln)

    # insert M block after the line that contains "potential root causes"
    insert_idx = None
    for i, ln in enumerate(cleaned_lines):
        if "potential root cause" in ln.lower():
            insert_idx = i + 1
            break

    if insert_idx is None:
        insert_idx = 0

    new_lines = cleaned_lines[:insert_idx] + [""] + m_lines + [""] + cleaned_lines[insert_idx:]
    return "\n".join(new_lines).strip()


def target_token_for_style_rca(question: str, canonical_cause: str) -> Optional[str]:
    """
    For RCA questions: get the label token used in THIS question for the given canonical cause.
    """
    m = build_cause_to_label_map(question)
    return m.get(canonical_cause)


# -------------------------
# F) Augment function per dataframe
# -------------------------
def augment_formats(df: pd.DataFrame,
                    styles: List[str] = ["plain", "letter", "P", "S", "I", "M5"],
                    ratio: float = 0.3,
                    per_row: int = 2,
                    seed: int = 7,
                    is_test: bool = False) -> pd.DataFrame:
    """
    - For train/val: keeps canonical_cause and sets answer token to match question format
    - For test: only augments questions; answer remains None
    """
    rnd = random.Random(seed)
    df = df.copy()

    if "ID" not in df.columns and "id" in df.columns:
        df["ID"] = df["id"]

    # add helper columns
    df["qtype"] = df["question"].apply(detect_question_type)
    df["answer_token"] = df["answer"].apply(extract_answer_token) if "answer" in df.columns else None

    # infer canonical cause for RCA in train/val
    if not is_test:
        def infer_cause(row):
            if row["qtype"] != "A_RCA":
                return None
            tok = row["answer_token"]
            if not tok:
                return None
            return infer_canonical_cause_from_answer(row["question"], tok)

        df["canonical_cause"] = df.apply(infer_cause, axis=1)
    else:
        df["canonical_cause"] = None

    # pick candidates to augment
    idxs = df.index.tolist()
    n_aug = int(len(df) * ratio)
    n_aug = min(n_aug, len(idxs))
    chosen = rnd.sample(idxs, n_aug) if n_aug > 0 else []

    augmented = []
    for idx in chosen:
        row = df.loc[idx].to_dict()
        q = row["question"]
        qtype = row["qtype"]
        base_id = row.get("ID", idx)

        # styles to apply
        st = styles[:]
        rnd.shuffle(st)
        st = st[:per_row]

        for style in st:
            q_new = q

            if style == "M5":
                # only meaningful for RCA
                if qtype != "A_RCA":
                    continue
                q_new = to_M5_question(q)
                if not q_new:
                    continue
            else:
                # relabel options in-place
                q_new = relabel_options_keep_order(q, style)

            out_row = {
                "ID": f"{base_id}_aug_{style}",
                "question": q_new,
                "answer": None,
            }

            if not is_test:
                # For train/val, compute answer token that matches THIS augmented question
                if qtype == "A_RCA":
                    cause = row.get("canonical_cause")
                    if cause:
                        tok = target_token_for_style_rca(q_new, cause)
                        out_row["answer"] = tok  # store token to be boxed later
                        out_row["canonical_cause"] = cause
                else:
                    # General MCQ: map by option position (if possible)
                    tok0 = row.get("answer_token")
                    opts0 = parse_options(q)
                    optsN = parse_options(q_new)
                    if tok0 and opts0 and optsN:
                        # find index of chosen option in original question
                        pos = None
                        for i2, o in enumerate(opts0, start=1):
                            if o["label"].strip() == tok0.strip():
                                pos = i2
                                break
                        if pos is not None and 1 <= pos <= len(optsN):
                            out_row["answer"] = optsN[pos-1]["label"]
            augmented.append(out_row)

    df_aug = pd.DataFrame(augmented)

    # Combine
    df_out = pd.concat([df, df_aug], ignore_index=True)

    print(f"✓ Augmented {len(df_aug)} rows (ratio={ratio}, per_row={per_row}, styles={styles})")
    return df_out


# -------------------------
# G) APPLY to your datasets
# -------------------------
# train/val have answers
train_df_aug = augment_formats(train_df, ratio=0.3, per_row=2, seed=11, is_test=False)
val_df_aug   = augment_formats(val_df,   ratio=0.3, per_row=2, seed=12, is_test=False)

# test has no answers (augment questions only)
test_df_aug  = test_df.copy()

# -------------------------
# H) SANITY CHECKS
# -------------------------
def check(df, name):
    print(f"\n--- {name} ---")
    print("rows:", len(df))
    print("qtype counts:\n", df["qtype"].value_counts(dropna=False))
    if "answer" in df.columns:
        missing = df["answer"].isna().sum()
        print("missing answers:", int(missing))
    # RCA rows where canonical_cause inferred but answer missing (train/val only)
    if "canonical_cause" in df.columns and "answer" in df.columns:
        bad = df[(df["qtype"]=="A_RCA") & (df["canonical_cause"].notna()) & (df["answer"].isna())]
        print("RCA rows with canonical_cause but no answer token:", len(bad))

check(train_df_aug, "train_df_aug")
check(val_df_aug,   "val_df_aug")
# check(test_df_aug,  "test_df_aug")

# Look at a couple augmented rows
display(train_df_aug[train_df_aug["ID"].astype(str).str.contains("_aug_")][["ID","qtype","answer","canonical_cause","question"]].head(2))


✓ Augmented 1440 rows (ratio=0.3, per_row=2, styles=['plain', 'letter', 'P', 'S', 'I', 'M5'])
✓ Augmented 518 rows (ratio=0.3, per_row=2, styles=['plain', 'letter', 'P', 'S', 'I', 'M5'])

--- train_df_aug ---
rows: 3840
qtype counts:
 qtype
A_RCA    2400
NaN      1440
Name: count, dtype: int64
missing answers: 110
RCA rows with canonical_cause but no answer token: 0

--- val_df_aug ---
rows: 1382
qtype counts:
 qtype
A_RCA    864
NaN      518
Name: count, dtype: int64
missing answers: 38
RCA rows with canonical_cause but no answer token: 0


,ID,qtype,answer,canonical_cause,question
2400,ID_HYIU6VU4N4_aug_P,NaN,P8,C8,Analyze the 5G wireless network drive-test use...
2401,ID_HYIU6VU4N4_aug_M5,NaN,M1,C8,Analyze the 5G wireless network drive-test use...


In [38]:
train_df_aug.tail()

,ID,question,answer,qtype,answer_token,canonical_cause
3835,ID_82QQW8T1LU_aug_M5,Analyze the 5G wireless network drive-test use...,M1,NaN,NaN,C8
3836,ID_G6RDCAKROH_aug_letter,Analyze the 5G wireless network drive-test use...,D,NaN,NaN,C4
3837,ID_G6RDCAKROH_aug_plain,Analyze the 5G wireless network drive-test use...,4,NaN,NaN,C4
3838,ID_200TTC3Y2J_aug_S,Analyze the 5G wireless network drive-test use...,S2,NaN,NaN,C2
3839,ID_200TTC3Y2J_aug_I,Analyze the 5G wireless network drive-test use...,I2,NaN,NaN,C2


In [40]:
print(train_df_aug['question'][3835])

Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{ans} in the final answer.

M1: Average scheduled RBs are below 160, affecting throughput.
M2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
M3: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
M4: Frequent handovers degrade performance.
M5: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.



Given:
- The default electronic downtilt value is 255, representing a downtilt angle of 6 degrees. Other values represent the actual downtilt angle in degrees.

Beam Scenario and Vertical Beamwidth Relationships:
- When the cell's Beam Scenario is set to Default or SCENARIO_1 to SCENARIO_5, the vertical beamwidth is 6 degrees.
- W

In [47]:
train_df[train_df['ID']=="ID_82QQW8T1LU"]

,ID,question,answer
1033,ID_82QQW8T1LU,Analyze the 5G wireless network drive-test use...,C8


In [49]:
print(train_df['question'][1033])

Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{ans} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user throughput.
C8: Average scheduled RBs are below 160, affecting throughput.

Given:
- The default electronic downtilt value is 255, representing a downtilt angle of 6 degrees. Other values represent the a

In [52]:
def to_float(x) -> Optional[float]:
    """Safely convert to float, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return float(x)
    except (ValueError, TypeError):
        return None

def to_int(x) -> Optional[int]:
    """Safely convert to int, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return int(float(x))  # float first to handle "123.0" strings
    except (ValueError, TypeError):
        return None

In [53]:
DRIVE_MAP = {
    "Timestamp": "timestamp",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "GPS Speed (km/h)": "gps_speed_kmh",
    "5G KPI PCell RF Serving PCI": "serving_pci",
    "5G KPI PCell RF Serving SS-RSRP [dBm]": "ss_rsrp_dbm",
    "5G KPI PCell RF Serving SS-SINR [dB]": "ss_sinr_db",
    "5G KPI PCell Layer2 MAC DL Throughput [Mbps]": "throughput_mbps",
    "5G KPI PCell Layer1 DL RB Num (Including 0)": "dl_rb_num",

    # Neighbor PCIs
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 PCI": "nei1_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 PCI": "nei2_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 PCI": "nei3_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 PCI": "nei4_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 PCI": "nei5_pci",

    # Neighbor RSRPs (CRITICAL - these were missing!)
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]": "nei1_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 Filtered Tx BRSRP [dBm]": "nei2_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 Filtered Tx BRSRP [dBm]": "nei3_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 Filtered Tx BRSRP [dBm]": "nei4_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 Filtered Tx BRSRP [dBm]": "nei5_rsrp_dbm",
}

# Engineering parameters column mappings
ENG_MAP = {
    "gNodeB ID": "gnodeb_id",
    "Cell ID": "cell_id",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Mechanical Azimuth": "mechanical_azimuth",
    "Mechanical Downtilt": "mechanical_downtilt",
    "Digital Tilt": "digital_tilt",
    "Digital Azimuth": "digital_azimuth",
    "Beam Scenario": "beam_scenario",
    "Height": "height",
    "PCI": "pci",
    "TxRx Mode": "txrx_mode",
    "Max Transmit Power": "max_tx_power",
    "Antenna Model": "antenna_model",
}

# Define fields by type for proper casting
FLOAT_FIELDS = [
    "longitude", "latitude", "gps_speed_kmh",
    "ss_rsrp_dbm", "ss_sinr_db", "throughput_mbps", "dl_rb_num",
    "mechanical_downtilt", "digital_tilt", "height", "max_tx_power",
    "mechanical_azimuth", "digital_azimuth",
    # CRITICAL: Add neighbor RSRP fields
    "nei1_rsrp_dbm", "nei2_rsrp_dbm", "nei3_rsrp_dbm",
    "nei4_rsrp_dbm", "nei5_rsrp_dbm",
]

INT_FIELDS = [
    "pci", "serving_pci",
    "nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci",
]

def normalize_rows(rows: List[Dict], mapping: Dict[str, str]) -> List[Dict]:
    """
    Normalize column names and apply type casting.
    This ensures all numeric operations work correctly.
    """
    normalized = []

    for r in rows:
        nr = {}

        # Map column names
        for k, v in r.items():
            normalized_key = mapping.get(k)
            if normalized_key is None:
                continue  # Skip unmapped columns
            nr[normalized_key] = v

        # Type casting - CRITICAL for numeric operations
        for field in FLOAT_FIELDS:
            if field in nr:
                nr[field] = to_float(nr[field])

        for field in INT_FIELDS:
            if field in nr:
                nr[field] = to_int(nr[field])

        normalized.append(nr)

    return normalized

In [50]:
def parse_pipe_table(table_text: str) -> List[Dict[str, Any]]:
    """
    Parse a pipe-delimited table into a list[dict].

    Works for common formats like:
      | Col A | Col B |
      |------:|:------|
      | 1     | 2     |

    Notes
    - Treats '-', '—', '–', '' as missing (None)
    - Ignores Markdown separator rows (e.g., '---', ':---:', '---:')
    """
    if not table_text or not str(table_text).strip():
        return []

    # keep only lines containing pipes
    raw_lines = [ln.strip() for ln in str(table_text).splitlines() if "|" in ln]
    if not raw_lines:
        return []

    def _is_separator_row(parts: List[str]) -> bool:
        # a separator row is made of dashes/colons only, e.g. '---', ':---:'
        def ok(p: str) -> bool:
            p = p.strip()
            return bool(p) and all(ch in "-: " for ch in p)
        return all(ok(p) for p in parts)

    # Normalize split: strip leading/trailing pipes then split
    def _split(line: str) -> List[str]:
        line = line.strip().strip("|")
        return [p.strip() for p in line.split("|")]

    header = _split(raw_lines[0])
    header = [h if h else f"col_{i}" for i, h in enumerate(header, start=1)]

    rows: List[Dict[str, Any]] = []
    for line in raw_lines[1:]:
        parts = _split(line)

        # drop Markdown separator lines
        if _is_separator_row(parts):
            continue

        # pad or trim to header length
        if len(parts) < len(header):
            parts += [""] * (len(header) - len(parts))
        elif len(parts) > len(header):
            parts = parts[: len(header)]

        rec = {}
        for k, v in zip(header, parts):
            v = v.strip()
            rec[k] = None if v in ("", "-", "—", "–") else v
        rows.append(rec)

    return rows


In [54]:
def compute_rca_features(drive_rows: List[Dict], eng_rows: List[Dict]) -> Dict[str, Any]:
    """
    Compute ONLY the features required by format_features_text.
    Optimized for efficiency - removed all unused calculations.
    
    Returns dict with 48 unique features used in the prompt.
    """
    features = {}
    
    # Helper: dBm -> mW
    def _dbm_to_mw(dbm_val: float) -> float:
        return 10 ** (dbm_val / 10.0)
    
    # Helper: safe correlation calculation
    def _safe_corr(xs, ys):
        xs2, ys2 = [], []
        for x, y in zip(xs, ys):
            if x is None or y is None:
                continue
            xs2.append(float(x))
            ys2.append(float(y))
        if len(xs2) < 2:
            return None
        mx, my = mean(xs2), mean(ys2)
        num = sum((x - mx) * (y - my) for x, y in zip(xs2, ys2))
        denx = sum((x - mx) ** 2 for x in xs2)
        deny = sum((y - my) ** 2 for y in ys2)
        if denx <= 0 or deny <= 0:
            return None
        return num / (math.sqrt(denx) * math.sqrt(deny))
    
    # -------------------------------------------------------------------------
    # C1 FEATURES: Excessive Downtilt
    # -------------------------------------------------------------------------
    serving_rsrps = [r["ss_rsrp_dbm"] for r in drive_rows if r.get("ss_rsrp_dbm") is not None]
    sinrs = [r["ss_sinr_db"] for r in drive_rows if r.get("ss_sinr_db") is not None]
    
    # rsrp_min_dbm, rsrp_mean_dbm, rsrp_10th_percentile
    if serving_rsrps:
        features["rsrp_min_dbm"] = min(serving_rsrps)
        features["rsrp_mean_dbm"] = mean(serving_rsrps)
        if len(serving_rsrps) > 2:
            features["rsrp_10th_percentile"] = sorted(serving_rsrps)[int(0.1 * len(serving_rsrps))]
        else:
            features["rsrp_10th_percentile"] = min(serving_rsrps)
    else:
        features["rsrp_min_dbm"] = None
        features["rsrp_mean_dbm"] = None
        features["rsrp_10th_percentile"] = None
    
    # sinr_degradation_db, sinr_std_when_rsrp_stable
    if serving_rsrps and sinrs and features["rsrp_mean_dbm"] is not None:
        sinr_mean = mean(sinrs)
        expected_sinr = features["rsrp_mean_dbm"] + 100
        features["sinr_degradation_db"] = expected_sinr - sinr_mean
        
        rsrp_std = stdev(serving_rsrps) if len(serving_rsrps) > 1 else 0.0
        sinr_std = stdev(sinrs) if len(sinrs) > 1 else 0.0
        denom = abs(rsrp_std) + 1e-6
        features["sinr_std_when_rsrp_stable"] = sinr_std / denom
    else:
        features["sinr_degradation_db"] = None
        features["sinr_std_when_rsrp_stable"] = None
    
    # handover_rsrp_delta_mean (computed later in handover section)
    
    # -------------------------------------------------------------------------
    # C2 FEATURES: Overshoot
    # -------------------------------------------------------------------------
    serving_pcis = [r["serving_pci"] for r in drive_rows if r.get("serving_pci") is not None]
    pci_to_cell = {c["pci"]: c for c in eng_rows if c.get("pci") is not None}
    
    # dist_max_m, dist_p95_m, overshoot_flag
    distances = []
    for r in drive_rows:
        spci = r.get("serving_pci")
        cell = pci_to_cell.get(spci)
        
        if not cell:
            continue
        
        if (r.get("latitude") is None or r.get("longitude") is None or
            cell.get("latitude") is None or cell.get("longitude") is None):
            continue
        
        dist = haversine_m(
            r["latitude"], r["longitude"],
            cell["latitude"], cell["longitude"]
        )
        distances.append(dist)
    
    if distances:
        features["dist_max_m"] = max(distances)
        features["dist_p95_m"] = sorted(distances)[int(0.95 * (len(distances) - 1))]
        features["overshoot_flag"] = features["dist_p95_m"] > 1000 if features["dist_p95_m"] else False
    else:
        features["dist_max_m"] = None
        features["dist_p95_m"] = None
        features["overshoot_flag"] = False
    
    # handover_count
    handover_count = 0
    for i in range(1, len(serving_pcis)):
        if serving_pcis[i] != serving_pcis[i - 1]:
            handover_count += 1
    features["handover_count"] = handover_count
    
    # throughput_variance_across_cells, best_cell_throughput_gap (computed later in C3)
    
    # -------------------------------------------------------------------------
    # C3 FEATURES: Wrong Cell Selection
    # -------------------------------------------------------------------------
    tps = [r["throughput_mbps"] for r in drive_rows if r.get("throughput_mbps") is not None]
    
    # tp_samples_below_600, tp_drop_ratio
    if tps:
        features["tp_samples_below_600"] = sum(1 for x in tps if x < 600)
        features["tp_drop_ratio"] = sum(1 for x in tps if x < 600) / len(tps)
    else:
        features["tp_samples_below_600"] = 0
        features["tp_drop_ratio"] = None
    
    # Handover TP deltas and cross-cell throughput comparison
    ho_tp_deltas = []
    ho_rsrp_deltas = []
    cell_throughputs = {}
    
    prev = None
    for r in drive_rows:
        # Track throughput by PCI
        pci = r.get("serving_pci")
        tp = r.get("throughput_mbps")
        if pci and tp:
            cell_throughputs.setdefault(pci, []).append(tp)
        
        # Handover deltas
        if prev is None:
            prev = r
            continue
        
        prev_pci = prev.get("serving_pci")
        curr_pci = r.get("serving_pci")
        
        if prev_pci is not None and curr_pci is not None and curr_pci != prev_pci:
            prev_tp = prev.get("throughput_mbps")
            curr_tp = r.get("throughput_mbps")
            prev_rsrp = prev.get("ss_rsrp_dbm")
            curr_rsrp = r.get("ss_rsrp_dbm")
            
            if prev_tp is not None and curr_tp is not None:
                ho_tp_deltas.append(curr_tp - prev_tp)
            if prev_rsrp is not None and curr_rsrp is not None:
                ho_rsrp_deltas.append(curr_rsrp - prev_rsrp)
        
        prev = r
    
    features["handover_tp_delta_mean"] = mean(ho_tp_deltas) if ho_tp_deltas else None
    features["handover_rsrp_delta_mean"] = mean(ho_rsrp_deltas) if ho_rsrp_deltas else None
    features["handover_improves_tp_flag"] = (features["handover_tp_delta_mean"] is not None and 
                                             features["handover_tp_delta_mean"] > 80)
    
    # avg_throughput_change_after_handover, throughput_improved_by_handover
    features["avg_throughput_change_after_handover"] = mean(ho_tp_deltas) if ho_tp_deltas else 0.0
    features["throughput_improved_by_handover"] = features["avg_throughput_change_after_handover"] > 50
    
    # throughput_variance_across_cells, best_cell_throughput_gap
    if len(cell_throughputs) >= 2:
        cell_avg_tp = {pci: mean(tps) for pci, tps in cell_throughputs.items()}
        
        if serving_pcis:
            most_common_pci = Counter(serving_pcis).most_common(1)[0][0]
            current_avg = cell_avg_tp.get(most_common_pci, 0)
            other_avgs = [avg for pci, avg in cell_avg_tp.items() if pci != most_common_pci]
            best_alt = max(other_avgs) if other_avgs else 0
            features["best_cell_throughput_gap"] = best_alt - current_avg
        else:
            features["best_cell_throughput_gap"] = 0.0
        
        best_tp_pci = max(cell_avg_tp.items(), key=lambda x: x[1])
        worst_tp_pci = min(cell_avg_tp.items(), key=lambda x: x[1])
        features["throughput_variance_across_cells"] = best_tp_pci[1] - worst_tp_pci[1]
    else:
        features["best_cell_throughput_gap"] = 0.0
        features["throughput_variance_across_cells"] = None
    
    # -------------------------------------------------------------------------
    # C4 FEATURES: Overlapping Coverage
    # -------------------------------------------------------------------------
    pci_to_gnodeb = {}
    for cell in eng_rows:
        if cell.get("pci") is not None:
            pci_to_gnodeb[cell["pci"]] = cell.get("gnodeb_id")
    
    # Get serving cell's gNodeB
    serving_gnodeb = None
    if serving_pcis:
        serving_pci = serving_pcis[-1]
        serving_gnodeb = pci_to_gnodeb.get(serving_pci)
    
    # noncolocated_strong_neighbor_gnodeb_count_mean, noncolocated_strong_neighbor_gnodeb_count_max
    noncolocated_strong_counts = []
    coloc_strong = 0
    noncoloc_strong = 0
    top1_top2_gaps = []
    
    neighbor_rsrp_by_pci = {}
    
    for r in drive_rows:
        serv_pci = r.get("serving_pci")
        serv_rsrp = r.get("ss_rsrp_dbm")
        if serv_pci is None or serv_rsrp is None:
            continue
        
        serv_gnb = pci_to_gnodeb.get(serv_pci)
        if serv_gnb is None:
            continue
        
        # Track strong non-colocated neighbors per sample
        strong_noncolocated_gnbs = set()
        nei_vals = []
        
        for i in range(1, 6):
            nei_pci = r.get(f"nei{i}_pci")
            nei_rsrp = r.get(f"nei{i}_rsrp_dbm")
            
            if nei_pci is None or nei_rsrp is None:
                continue
            
            # Track for neighbor analysis
            if nei_pci not in neighbor_rsrp_by_pci:
                neighbor_rsrp_by_pci[nei_pci] = []
            neighbor_rsrp_by_pci[nei_pci].append(nei_rsrp)
            nei_vals.append(nei_rsrp)
            
            # Strong neighbor: RSRP > -95 AND within 6 dB of serving
            if nei_rsrp > -95 and abs(nei_rsrp - serv_rsrp) <= 6:
                nei_gnb = pci_to_gnodeb.get(nei_pci)
                if nei_gnb is not None:
                    if nei_gnb != serv_gnb:
                        strong_noncolocated_gnbs.add(nei_gnb)
                        noncoloc_strong += 1
                    else:
                        coloc_strong += 1
        
        noncolocated_strong_counts.append(len(strong_noncolocated_gnbs))
        
        # top1_top2_gap_db_mean
        if len(nei_vals) >= 2:
            sorted_nei = sorted(nei_vals, reverse=True)
            top1, top2 = sorted_nei[0], sorted_nei[1]
            top1_top2_gaps.append(top1 - top2)
    
    features["noncolocated_strong_neighbor_gnodeb_count_mean"] = (mean(noncolocated_strong_counts) 
                                                                    if noncolocated_strong_counts else 0.0)
    features["noncolocated_strong_neighbor_gnodeb_count_max"] = (max(noncolocated_strong_counts) 
                                                                  if noncolocated_strong_counts else 0)
    features["top1_top2_gap_db_mean"] = mean(top1_top2_gaps) if top1_top2_gaps else None
    
    # strong_neighbor_noncolocated_share
    denom = coloc_strong + noncoloc_strong
    features["strong_neighbor_noncolocated_share"] = (noncoloc_strong / denom) if denom > 0 else 0.0
    
    # high_interference_power_ratio_flag
    ratios_db = []
    for r in drive_rows:
        serv_rsrp = r.get("ss_rsrp_dbm")
        if serv_rsrp is None:
            continue
        S = _dbm_to_mw(serv_rsrp)
        I = 0.0
        for i in range(1, 6):
            nei_rsrp = r.get(f"nei{i}_rsrp_dbm")
            if nei_rsrp is None:
                continue
            if nei_rsrp > -95 and abs(nei_rsrp - serv_rsrp) <= 6:
                I += _dbm_to_mw(nei_rsrp)
        if S > 0 and I > 0:
            ratios_db.append(10.0 * math.log10((I + 1e-12) / (S + 1e-12)))
    
    if ratios_db:
        ratios_db_sorted = sorted(ratios_db)
        p90_idx = int(0.9 * (len(ratios_db_sorted) - 1))
        features["high_interference_power_ratio_flag"] = ratios_db_sorted[p90_idx] > -3.0
    else:
        features["high_interference_power_ratio_flag"] = False
    
    # rsrp_advantage_of_best_neighbor
    best_neighbor_rsrp = None
    if neighbor_rsrp_by_pci:
        best_pci = max(neighbor_rsrp_by_pci.items(), key=lambda x: mean(x[1]))
        best_neighbor_rsrp = mean(best_pci[1])
    
    serving_rsrp_avg = mean(serving_rsrps) if serving_rsrps else None
    if best_neighbor_rsrp is not None and serving_rsrp_avg is not None:
        features["rsrp_advantage_of_best_neighbor"] = best_neighbor_rsrp - serving_rsrp_avg
    else:
        features["rsrp_advantage_of_best_neighbor"] = None
    
    # -------------------------------------------------------------------------
    # C5 FEATURES: Ping-Pong Handover
    # -------------------------------------------------------------------------
    # ping_pong_handover_count, ping_pong_detected, frequent_handover_flag
    ping_pong_events = 0
    prev_pci = None
    prev_prev_pci = None
    
    for r in drive_rows:
        curr_pci = r.get("serving_pci")
        if prev_pci and prev_prev_pci and curr_pci == prev_prev_pci and curr_pci != prev_pci:
            ping_pong_events += 1
        prev_prev_pci = prev_pci
        prev_pci = curr_pci
    
    features["ping_pong_handover_count"] = ping_pong_events
    features["ping_pong_detected"] = ping_pong_events >= 2
    features["frequent_handover_flag"] = handover_count >= 3
    
    # Note: handover_count, handover_rsrp_delta_mean, handover_tp_delta_mean already computed above
    
    # -------------------------------------------------------------------------
    # C6 FEATURES: PCI Collision
    # -------------------------------------------------------------------------
    # serving_electronic_tilt_deg, serving_total_tilt_deg, tilt_to_beamwidth_ratio
    # noncolocated_neighbor_count, colocated_neighbor_count, abnormal_path_loss
    
    if serving_pcis:
        cell = pci_to_cell.get(serving_pcis[-1])
        
        if cell:
            mech_tilt = cell.get("mechanical_downtilt") or 0.0
            elec_tilt = electronic_tilt_deg(cell.get("digital_tilt"))
            total_tilt = float(mech_tilt) + float(elec_tilt)
            
            vbw = beamwidth_deg(cell.get("beam_scenario", "DEFAULT"))
            tilt_to_beamwidth_ratio = total_tilt / vbw if vbw > 0 else 0.0
            
            features["serving_electronic_tilt_deg"] = elec_tilt
            features["serving_total_tilt_deg"] = total_tilt
            features["tilt_to_beamwidth_ratio"] = tilt_to_beamwidth_ratio
        else:
            features["serving_electronic_tilt_deg"] = None
            features["serving_total_tilt_deg"] = None
            features["tilt_to_beamwidth_ratio"] = None
    else:
        features["serving_electronic_tilt_deg"] = None
        features["serving_total_tilt_deg"] = None
        features["tilt_to_beamwidth_ratio"] = None
    
    # noncolocated_neighbor_count, colocated_neighbor_count
    neighbor_gnodebs = set()
    colocated_neighbors = []
    noncolocated_neighbors = []
    
    for r in drive_rows:
        for k in ["nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci"]:
            nei_pci = r.get(k)
            if nei_pci is not None:
                nei_gnodeb = pci_to_gnodeb.get(nei_pci)
                if nei_gnodeb is not None:
                    neighbor_gnodebs.add(nei_gnodeb)
                    if serving_gnodeb is not None:
                        if nei_gnodeb == serving_gnodeb:
                            colocated_neighbors.append(nei_pci)
                        else:
                            noncolocated_neighbors.append(nei_pci)
    
    features["noncolocated_neighbor_count"] = len(set(noncolocated_neighbors))
    features["colocated_neighbor_count"] = len(set(colocated_neighbors))
    
    # abnormal_path_loss
    if distances and serving_rsrps and len(distances) == len(serving_rsrps):
        path_loss_deviations = []
        for i, (dist, rsrp) in enumerate(zip(distances, serving_rsrps)):
            if dist > 10:
                dist_km = dist / 1000
                if dist_km > 0:
                    expected_pl = 128.1 + 37.6 * math.log10(dist_km)
                    actual_pl = 46 - rsrp
                    deviation = abs(actual_pl - expected_pl)
                    path_loss_deviations.append(deviation)
        
        if path_loss_deviations:
            path_loss_deviation_mean = mean(path_loss_deviations)
            features["abnormal_path_loss"] = path_loss_deviation_mean > 15
        else:
            features["abnormal_path_loss"] = False
    else:
        features["abnormal_path_loss"] = False
    
    # -------------------------------------------------------------------------
    # C7 FEATURES: High Speed
    # -------------------------------------------------------------------------
    speeds = [r["gps_speed_kmh"] for r in drive_rows if r.get("gps_speed_kmh") is not None]
    
    if speeds:
        features["speed_max_kmh"] = max(speeds)
        features["speed_mean_kmh"] = mean(speeds)
        features["speed_above_40_flag"] = features["speed_max_kmh"] > 40 if features["speed_max_kmh"] else False
    else:
        features["speed_max_kmh"] = None
        features["speed_mean_kmh"] = None
        features["speed_above_40_flag"] = False
    
    # fast_low_tp_ratio, speed_tp_correlation, c7_speed_hint
    speed_tp_pairs = []
    fast_total = 0
    fast_low_tp = 0
    
    for r in drive_rows:
        spd = r.get("gps_speed_kmh")
        tp = r.get("throughput_mbps")
        if spd is None or tp is None:
            continue
        speed_tp_pairs.append((spd, tp))
        if spd >= 40:
            fast_total += 1
            if tp < 600:
                fast_low_tp += 1
    
    features["fast_low_tp_ratio"] = (fast_low_tp / fast_total) if fast_total > 0 else 0.0
    
    if speed_tp_pairs:
        speeds2 = [x for x, _ in speed_tp_pairs]
        tps2 = [y for _, y in speed_tp_pairs]
        features["speed_tp_correlation"] = _safe_corr(speeds2, tps2)
    else:
        features["speed_tp_correlation"] = None
    
    features["c7_speed_hint"] = (
        features.get("speed_above_40_flag", False) and
        features.get("fast_low_tp_ratio", 0.0) > 0.25
    )
    
    # -------------------------------------------------------------------------
    # C8 FEATURES: Resource Congestion
    # -------------------------------------------------------------------------
    rbs = [r["dl_rb_num"] for r in drive_rows if r.get("dl_rb_num") is not None]
    
    if rbs:
        features["rb_mean"] = mean(rbs)
        features["rb_min"] = min(rbs)
    else:
        features["rb_mean"] = None
        features["rb_min"] = None
    
    # high_rb_low_tp_ratio_v2, tp_drop_with_high_rb_ratio, rb_tp_correlation
    rb_tp_pairs2 = []
    high_rb_total = 0
    high_rb_low_tp = 0
    drop_total = 0
    drop_high_rb = 0
    
    for r in drive_rows:
        rb = r.get("dl_rb_num")
        tp = r.get("throughput_mbps")
        if rb is None or tp is None:
            continue
        
        rb_tp_pairs2.append((rb, tp))
        
        if rb >= 180:
            high_rb_total += 1
            if tp < 600:
                high_rb_low_tp += 1
        
        if tp < 600:
            drop_total += 1
            if rb > 180:
                drop_high_rb += 1
    
    features["high_rb_low_tp_ratio_v2"] = (high_rb_low_tp / high_rb_total) if high_rb_total > 0 else 0.0
    features["tp_drop_with_high_rb_ratio"] = (drop_high_rb / drop_total) if drop_total > 0 else 0.0
    
    if rb_tp_pairs2:
        rbs2 = [x for x, _ in rb_tp_pairs2]
        tps3 = [y for _, y in rb_tp_pairs2]
        features["rb_tp_correlation"] = _safe_corr(rbs2, tps3)
    else:
        features["rb_tp_correlation"] = None
    
    # throughput_efficiency_min
    tp_rb_pairs = [(r.get("throughput_mbps"), r.get("dl_rb_num"))
                   for r in drive_rows
                   if r.get("throughput_mbps") is not None and r.get("dl_rb_num") is not None and r.get("dl_rb_num") > 0]
    
    if tp_rb_pairs:
        efficiencies = [tp / rb for tp, rb in tp_rb_pairs]
        features["throughput_efficiency_min"] = min(efficiencies)
    else:
        features["throughput_efficiency_min"] = None
    
    return features



In [59]:
# ============================================================================
# DOMAIN-SPECIFIC CALCULATIONS
# ============================================================================

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance in meters between two lat/lon points"""
    R = 6371000.0  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def beamwidth_deg(beam_scenario: str) -> int:
    """
    Map beam scenario to vertical beamwidth:
    - DEFAULT/SCENARIO_1-5: 6 degrees
    - SCENARIO_6-11: 12 degrees
    - SCENARIO_12+: 25 degrees
    """
    s = (beam_scenario or "").upper().strip()
    if s == "DEFAULT":
        return 6

    m = re.match(r"SCENARIO_(\d+)", s)
    if not m:
        return 6

    k = int(m.group(1))
    if 1 <= k <= 5:
        return 6
    elif 6 <= k <= 11:
        return 12
    else:  # 12+
        return 25

def electronic_tilt_deg(digital_tilt) -> float:
    """
    Convert digital tilt to degrees:
    - 255 => 6 degrees (special case)
    - Otherwise: value is degrees
    """
    try:
        v = int(float(digital_tilt))
        return 6.0 if v == 255 else float(v)
    except (ValueError, TypeError):
        return 6.0  # Default fallback


In [60]:
def format_value(v) -> str:
    """Format value for display in prompt"""
    if v is None:
        return "N/A"
    if isinstance(v, bool):
        return "Yes" if v else "No"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return f"{v:.2f}"
    return str(v)

def format_features_text(features: Dict) -> str:
    """
    Compact format focusing on top discriminating features per class.
    Optimized for token efficiency while preserving classification power.
    """
    feature_text = "\n".join([
        "**Key RCA Features:**",
        "",
        "**C1 Indicators (Excessive Downtilt):**",
        "  1. RSRP min: {0} dBm".format(format_value(features.get('rsrp_min_dbm'))),
        "  2. RSRP 10th percentile: {0} dBm".format(format_value(features.get('rsrp_10th_percentile'))),
        "  3. Handover RSRP delta mean: {0} dB".format(format_value(features.get('handover_rsrp_delta_mean'))),
        "  4. RSRP mean: {0} dBm".format(format_value(features.get('rsrp_mean_dbm'))),
        "  5. SINR degradation: {0} dB".format(format_value(features.get('sinr_degradation_db'))),
        "  6. SINR std when RSRP stable: {0}".format(format_value(features.get('sinr_std_when_rsrp_stable'))),
        "",
        "**C2 Indicators (Overshoot):**",
        "  1. Distance max: {0} m".format(format_value(features.get('dist_max_m'))),
        "  2. Overshoot flag: {0}".format(format_value(features.get('overshoot_flag'))),
        "  3. Distance p95: {0} m".format(format_value(features.get('dist_p95_m'))),
        "  4. TP variance across cells: {0}".format(format_value(features.get('throughput_variance_across_cells'))),
        "  5. Handover count: {0}".format(format_value(features.get('handover_count'))),
        "  6. Best cell TP gap: {0} Mbps".format(format_value(features.get('best_cell_throughput_gap'))),
        "",
        "**C3 Indicators (Wrong Cell Selection):**",
        "  1. Avg TP change after handover: {0} Mbps".format(format_value(features.get('avg_throughput_change_after_handover'))),
        "  2. Handover TP delta mean: {0} Mbps".format(format_value(features.get('handover_tp_delta_mean'))),
        "  3. Handover improves TP flag: {0}".format(format_value(features.get('handover_improves_tp_flag'))),
        "  4. TP improved by handover: {0}".format(format_value(features.get('throughput_improved_by_handover'))),
        "  5. TP samples below 600: {0}".format(format_value(features.get('tp_samples_below_600'))),
        "  6. TP drop ratio: {0}".format(format_value(features.get('tp_drop_ratio'))),
        "",
        "**C4 Indicators (Overlapping Coverage):**",
        "  1. Non-colocated strong neighbor gNodeB count mean: {0}".format(format_value(features.get('noncolocated_strong_neighbor_gnodeb_count_mean'))),
        "  2. Strong neighbor non-colocated share: {0}".format(format_value(features.get('strong_neighbor_noncolocated_share'))),
        "  3. Non-colocated strong neighbor gNodeB count max: {0}".format(format_value(features.get('noncolocated_strong_neighbor_gnodeb_count_max'))),
        "  4. High interference power ratio flag: {0}".format(format_value(features.get('high_interference_power_ratio_flag'))),
        "  5. RSRP advantage of best neighbor: {0} dB".format(format_value(features.get('rsrp_advantage_of_best_neighbor'))),
        "  6. Top1-top2 gap dB mean: {0} dB".format(format_value(features.get('top1_top2_gap_db_mean'))),
        "",
        "**C5 Indicators (Ping-Pong Handover):**",
        "  1. Ping-pong handover count: {0}".format(format_value(features.get('ping_pong_handover_count'))),
        "  2. Frequent handover flag: {0}".format(format_value(features.get('frequent_handover_flag'))),
        "  3. Ping-pong detected: {0}".format(format_value(features.get('ping_pong_detected'))),
        "  4. Handover count: {0}".format(format_value(features.get('handover_count'))),
        "  5. Handover RSRP delta mean: {0} dB".format(format_value(features.get('handover_rsrp_delta_mean'))),
        "  6. Handover TP delta mean: {0} Mbps".format(format_value(features.get('handover_tp_delta_mean'))),
        "",
        "**C6 Indicators (PCI Collision):**",
        "  1. Serving electronic tilt: {0}°".format(format_value(features.get('serving_electronic_tilt_deg'))),
        "  2. Serving total tilt: {0}°".format(format_value(features.get('serving_total_tilt_deg'))),
        "  3. Non-colocated neighbor count: {0}".format(format_value(features.get('noncolocated_neighbor_count'))),
        "  4. Abnormal path loss: {0}".format(format_value(features.get('abnormal_path_loss'))),
        "  5. Tilt to beamwidth ratio: {0}".format(format_value(features.get('tilt_to_beamwidth_ratio'))),
        "  6. Co-located neighbor count: {0}".format(format_value(features.get('colocated_neighbor_count'))),
        "",
        "**C7 Indicators (High Speed):**",
        "  1. Speed max: {0} km/h".format(format_value(features.get('speed_max_kmh'))),
        "  2. C7 speed hint: {0}".format(format_value(features.get('c7_speed_hint'))),
        "  3. Speed above 40 flag: {0}".format(format_value(features.get('speed_above_40_flag'))),
        "  4. Fast low TP ratio: {0}".format(format_value(features.get('fast_low_tp_ratio'))),
        "  5. Speed mean: {0} km/h".format(format_value(features.get('speed_mean_kmh'))),
        "  6. Speed-TP correlation: {0}".format(format_value(features.get('speed_tp_correlation'))),
        "",
        "**C8 Indicators (Resource Congestion):**",
        "  1. RB mean: {0}".format(format_value(features.get('rb_mean'))),
        "  2. High RB low TP ratio v2: {0}".format(format_value(features.get('high_rb_low_tp_ratio_v2'))),
        "  3. RB min: {0}".format(format_value(features.get('rb_min'))),
        "  4. TP drop with high RB ratio: {0}".format(format_value(features.get('tp_drop_with_high_rb_ratio'))),
        "  5. RB-TP correlation: {0}".format(format_value(features.get('rb_tp_correlation'))),
        "  6. TP efficiency min: {0}".format(format_value(features.get('throughput_efficiency_min'))),
    ])

    return feature_text

In [61]:
import re
import pandas as pd
from typing import Dict, Any, Optional, Tuple

# -------------------------
# A) Markers (robust to spelling + colon types)
# -------------------------
DRIVE_PAT = re.compile(r"User\s+plane\s+drive\s+test\s+data\s+as\s+follows\s*[:：]\s*", re.IGNORECASE)
ENG_PAT   = re.compile(r"(?:Engeneering|Engineering)\s+parameters\s+data\s+as\s+follows\s*[:：]\s*", re.IGNORECASE)

def detect_question_type(question: str) -> str:
    q = question or ""
    ql = q.lower()
    if DRIVE_PAT.search(q) and ENG_PAT.search(q):
        return "A_RCA"
    keywords = ["rsrp", "sinr", "throughput", "handover", "gnodeb", "rb", "pci", "drive test", "engineering parameters"]
    return "A_RCA" if any(k in ql for k in keywords) else "B_GENERAL"


# -------------------------
# B) Section extractor that works in BOTH orders
# -------------------------
def extract_drive_and_eng_sections(question_text: str) -> Optional[Tuple[str, str]]:
    """
    Returns (drive_text, eng_text) as raw strings.
    Works whether Engineering appears before Drive or vice versa.
    """
    if not question_text:
        return None

    d = DRIVE_PAT.search(question_text)
    e = ENG_PAT.search(question_text)
    if not d or not e:
        return None

    # Determine order by start positions
    if d.start() < e.start():
        # Drive first, then Eng
        drive_text = question_text[d.end():e.start()].strip()
        eng_text   = question_text[e.end():].strip()
    else:
        # Eng first, then Drive
        eng_text   = question_text[e.end():d.start()].strip()
        drive_text = question_text[d.end():].strip()

    return drive_text, eng_text


# -------------------------
# C) Compute features for one row (RCA only)
# -------------------------
def compute_features_from_question(question_text: str) -> Dict[str, Any]:
    """
    Returns a dict with:
      - parse_ok (bool)
      - parse_error (str|None)
      - drive_nrows, eng_nrows
      - features_dict (dict|None)
      - features_text (str|None)
    """
    out = {
        "parse_ok": False,
        "parse_error": None,
        "drive_nrows": 0,
        "eng_nrows": 0,
        "features_dict": None,
        "features_text": None,
    }

    sec = extract_drive_and_eng_sections(question_text)
    if not sec:
        out["parse_error"] = "Missing Drive/Engineering markers"
        return out

    drive_text, eng_text = sec

    drive_raw = parse_pipe_table(drive_text)
    eng_raw   = parse_pipe_table(eng_text)

    if not drive_raw:
        out["parse_error"] = "Drive table parse returned 0 rows"
        return out
    if not eng_raw:
        out["parse_error"] = "Engineering table parse returned 0 rows"
        return out

    drive_rows = normalize_rows(drive_raw, DRIVE_MAP)
    eng_rows   = normalize_rows(eng_raw, ENG_MAP)

    out["drive_nrows"] = len(drive_rows)
    out["eng_nrows"]   = len(eng_rows)

    feats = compute_rca_features(drive_rows, eng_rows)
    out["features_dict"] = feats
    out["features_text"] = format_features_text(feats)
    out["parse_ok"] = True
    return out


# -------------------------
# D) Apply to a whole dataframe (adds columns)
# -------------------------
def add_rca_features(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.copy()

    # ensure qtype exists
    if "qtype" not in df.columns:
        df["qtype"] = df["question"].astype(str).apply(detect_question_type)

    # init columns
    for col in ["parse_ok", "parse_error", "drive_nrows", "eng_nrows", "features_dict", "features_text"]:
        if col not in df.columns:
            df[col] = None

    # compute only for RCA rows
    mask = df["qtype"] == "A_RCA"

    results = df.loc[mask, "question"].apply(compute_features_from_question)
    # results is a series of dicts -> expand
    results_df = pd.DataFrame(list(results.values), index=results.index)

    for col in results_df.columns:
        df.loc[results_df.index, col] = results_df[col]

    # quick report
    total = len(df)
    rca = int(mask.sum())
    ok = int((df.loc[mask, "parse_ok"] == True).sum())
    bad = rca - ok
    print(f"✓ {name}: total={total} | RCA={rca} | parsed_ok={ok} | failed={bad}")

    if bad > 0:
        print("Top failure reasons:")
        print(df.loc[mask & (df["parse_ok"] != True), "parse_error"].value_counts().head(10))

    return df


# -------------------------
# E) Run on your datasets
# -------------------------
train_df_aug = add_rca_features(train_df_aug, "train_df_aug")
val_df_aug   = add_rca_features(val_df_aug,   "val_df_aug")

# test remains original (no augmentation) but we still want features for RCA questions
test_df      = add_rca_features(test_df,      "test_df")

# Inspect a successful example
display(train_df_aug[(train_df_aug["qtype"]=="A_RCA") & (train_df_aug["parse_ok"]==True)][
    ["ID","drive_nrows","eng_nrows","features_text"]
].head(1))

# Inspect failures if any
display(train_df_aug[(train_df_aug["qtype"]=="A_RCA") & (train_df_aug["parse_ok"]!=True)][
    ["ID","parse_error","question"]
].head(1))


✓ train_df_aug: total=3840 | RCA=2400 | parsed_ok=2400 | failed=0
✓ val_df_aug: total=1382 | RCA=864 | parsed_ok=864 | failed=0
✓ test_df: total=863 | RCA=784 | parsed_ok=681 | failed=103
Top failure reasons:
parse_error
Missing Drive/Engineering markers    103
Name: count, dtype: int64


,ID,drive_nrows,eng_nrows,features_text
0,ID_1P7PJMPV0R,10,5,**Key RCA Features:**\n\n**C1 Indicators (Excessive Downtilt):**\n 1. RSRP min: -88.21 dBm\n 2. RSRP 10th percentile: -85.50 dBm\n 3. Handover RSRP delta mean: -1.48 dB\n 4. RSRP mean: -79.36 ...


,ID,parse_error,question


In [63]:
def safe_format(val, unit="", decimals=1):
    """Safely format feature values, handling None/missing data"""
    if val is None:
        return "N/A"
    try:
        if isinstance(val, (int, float)):
            if decimals == 0:
                return f"{int(val)}{unit}"
            return f"{float(val):.{decimals}f}{unit}"
        return str(val)
    except (ValueError, TypeError):
        return "N/A"

def get_relevant_features(answer: str, features: Dict) -> Dict:
    """Extract only features relevant to specific root cause class"""
    # Common features for all classes
    common = {
        'rsrp_mean_dbm': features.get('rsrp_mean_dbm'),
        'sinr_mean_db': features.get('sinr_mean_db')
    }
    
    # Class-specific feature subsets
    class_features = {
        "C1": ['rsrp_min_dbm', 'rsrp_10th_percentile', 'sinr_degradation_db', 
               'serving_total_tilt_deg', 'dist_p95_m', 'rsrp_advantage_of_best_neighbor'],
        "C2": ['dist_max_m', 'dist_p95_m', 'overshoot_flag', 'handover_count',
               'throughput_variance_across_cells', 'best_cell_throughput_gap'],
        "C3": ['avg_throughput_change_after_handover', 'handover_tp_delta_mean',
               'handover_improves_tp_flag', 'tp_samples_below_600', 'tp_drop_ratio'],
        "C4": ['noncolocated_strong_neighbor_gnodeb_count_mean', 'strong_neighbor_noncolocated_share',
               'top1_top2_gap_db_mean', 'high_interference_power_ratio_flag', 'rsrp_advantage_of_best_neighbor'],
        "C5": ['ping_pong_handover_count', 'frequent_handover_flag', 'ping_pong_detected',
               'handover_count', 'handover_rsrp_delta_mean', 'handover_tp_delta_mean'],
        "C6": ['noncolocated_neighbor_count', 'colocated_neighbor_count', 'serving_pci',
               'abnormal_path_loss', 'serving_electronic_tilt_deg'],
        "C7": ['speed_max_kmh', 'c7_speed_hint', 'speed_above_40_flag',
               'fast_low_tp_ratio', 'speed_mean_kmh', 'speed_tp_correlation'],
        "C8": ['rb_mean', 'high_rb_low_tp_ratio_v2', 'tp_drop_with_high_rb_ratio',
               'rb_tp_correlation', 'throughput_efficiency_min', 'rb_min']
    }
    
    relevant = common.copy()
    for feat in class_features.get(answer, []):
        if feat in features:
            relevant[feat] = features[feat]
    
    return relevant

def format_optimal_training_prompt(row: Dict) -> Dict:
    """
    Creates optimal training format for Qwen 1.5B with:
    - COMPACT system prompt (~100 tokens)
    - BRIEF reasoning (max 150 tokens)
    - FILTERED features (only relevant per class)
    - Clear decision rules
    """

    # Extract the engineered features
    features = row['features_dict']
    original_q = row['original_question']
    answer = row['answer']

    # Build COMPACT system prompt optimized for 1.5B model (~100 tokens)
    EXPERT_SYSTEM = """You are a 5G RAN specialist. Analyze systematically:
1. RSRP/SINR patterns 2. Neighbor relations 3. Handover behavior 4. Speed/resources

Classes: C1=Downtilt (weak edge), C2=Overshoot (>1km), C3=WrongCell (HO improves TP), C4=Interference (good RSRP+poor SINR), C5=PingPong (frequent HO), C6=PCI collision, C7=Speed (>40km/h), C8=Congestion (high RB+low TP)

Output: Brief reasoning + \\boxed{{n}}"""

    # Build reasoning-enhanced user prompt
    USER_PROMPT = f"""{original_q}

{row['features_text']}

**Analysis Instructions:**
Think step-by-step:

1. **Signal Quality Assessment:**
   - What are the RSRP and SINR levels?
   - Are they correlated (both low = C1/C3) or decoupled (good RSRP + poor SINR = C4)?

2. **Neighbor Analysis:**
   - How many strong neighbors exist?
   - Are they from the same gNodeB (co-located) or different sites (non-co-located)?
   - Is there one dominant neighbor (C3) or multiple equal neighbors (C4)?

3. **Mobility & Handover:**
   - How many handovers occurred?
   - Do handovers improve throughput (C3/C5) or not?
   - Is there ping-pong behavior (back-and-forth)?

4. **Efficiency Metrics:**
   - Is RB allocation high but throughput low (C8)?
   - Does speed correlate with poor performance (C7)?
   - Is there abnormal path loss or tilt issues (C1)?

5. **Distance & Coverage:**
   - Is the UE very far from the cell (C2)?
   - Are there coverage holes or weak spots (C1)?

Based on this systematic analysis, identify the PRIMARY root cause."""

    # Build reasoning-based assistant response with COMPACT formatting
    ASSISTANT_REASONING = generate_reasoning_for_class(answer, features)

    ASSISTANT_RESPONSE = f"""{ASSISTANT_REASONING}

**Final Answer:** \\boxed{{{answer[1]}}}"""  # Extract number from "C1" -> "1"

    return {
        "messages": [
            {"role": "system", "content": EXPERT_SYSTEM},
            {"role": "user", "content": USER_PROMPT},
            {"role": "assistant", "content": ASSISTANT_RESPONSE}
        ]
    }


def generate_reasoning_for_class(answer: str, features: Dict) -> str:
    """
    Generate COMPACT class-specific reasoning (max 150 tokens).
    Optimized for 1.5B model - filters only relevant features.
    """
    # Filter to only relevant features for this class
    relevant = get_relevant_features(answer, features)
    
    # Extract key metrics with safe formatting
    rsrp_mean = safe_format(relevant.get('rsrp_mean_dbm'), ' dBm')
    sinr_mean = safe_format(relevant.get('sinr_mean_db'), ' dB')

    # COMPACT reasoning templates (max 150 tokens each)
    reasoning_templates = {
        "C1": f"""**Analysis:**
• Signal: RSRP={rsrp_mean} (weak edge), SINR={sinr_mean}
• Coverage: RSRP min={safe_format(relevant.get('rsrp_min_dbm'), ' dBm')}, p10={safe_format(relevant.get('rsrp_10th_percentile'), ' dBm')} → poor far-end
• Geometry: Tilt={safe_format(relevant.get('serving_total_tilt_deg'), '°')}, dist_p95={safe_format(relevant.get('dist_p95_m'), 'm')}
• SINR degrades {safe_format(relevant.get('sinr_degradation_db'), ' dB')} when RSRP stable
• No dominant better neighbor (advantage: {safe_format(relevant.get('rsrp_advantage_of_best_neighbor'), ' dB')})
**Root Cause:** C1 - Excessive downtilt causing weak far coverage""",

        "C2": f"""**Analysis:**
• Distance: max={safe_format(relevant.get('dist_max_m'), 'm')}, p95={safe_format(relevant.get('dist_p95_m'), 'm')} → overshoot (>1000m)
• Overshoot flag: {safe_format(relevant.get('overshoot_flag'))}
• Handovers: {safe_format(relevant.get('handover_count'), '', 0)} times
• TP variance across cells: {safe_format(relevant.get('throughput_variance_across_cells'), ' Mbps')} → multiple cells visible
• Best cell TP gap: {safe_format(relevant.get('best_cell_throughput_gap'), ' Mbps')}
**Root Cause:** C2 - Overshoot (serving cell too far, coverage boundary issue)""",

        "C3": f"""**Analysis:**
• Handover impact: TP change after HO = {safe_format(relevant.get('avg_throughput_change_after_handover'), ' Mbps')} (significant improvement)
• HO TP delta: {safe_format(relevant.get('handover_tp_delta_mean'), ' Mbps')} → improves throughput: {safe_format(relevant.get('handover_improves_tp_flag'))}
• Poor TP: {safe_format(relevant.get('tp_samples_below_600'), '', 0)} samples <600 Mbps, drop ratio={safe_format(relevant.get('tp_drop_ratio'))}
• Signal: RSRP={rsrp_mean}, SINR={sinr_mean} (both degraded, correlated)
• Pattern: Single dominant neighbor available (not C4 symmetric interference)
**Root Cause:** C3 - Wrong cell selection (handovers consistently improve TP)""",

        "C4": f"""**Analysis:**
• Signal decoupling: RSRP={rsrp_mean} (good), SINR={sinr_mean} (poor) → interference
• Multi-site: Non-colocated strong neighbors={safe_format(relevant.get('noncolocated_strong_neighbor_gnodeb_count_mean'))}
• Non-colocated share: {safe_format(relevant.get('strong_neighbor_noncolocated_share'))}
• Symmetry: Top1-top2 gap={safe_format(relevant.get('top1_top2_gap_db_mean'), ' dB')} (small) → multiple equal competitors
• High interference flag: {safe_format(relevant.get('high_interference_power_ratio_flag'))}
• Best neighbor advantage: {safe_format(relevant.get('rsrp_advantage_of_best_neighbor'), ' dB')} (no clear winner)
**Root Cause:** C4 - Overlapping coverage (multiple non-colocated sites, symmetric interference)""",

        "C5": f"""**Analysis:**
• Ping-pong: {safe_format(relevant.get('ping_pong_handover_count'), '', 0)} detected events
• Frequent HO flag: {safe_format(relevant.get('frequent_handover_flag'))}, ping-pong detected: {safe_format(relevant.get('ping_pong_detected'))}
• Total handovers: {safe_format(relevant.get('handover_count'), '', 0)} (≥3 = frequent)
• HO RSRP delta: {safe_format(relevant.get('handover_rsrp_delta_mean'), ' dB')}
• HO TP delta: {safe_format(relevant.get('handover_tp_delta_mean'), ' Mbps')}
• Pattern: Back-and-forth switching (not unidirectional mobility)
**Root Cause:** C5 - Ping-pong handover (hysteresis/parameter issue)""",

        "C6": f"""**Analysis:**
• PCI config: Serving PCI={safe_format(relevant.get('serving_pci'), '', 0)}
• Neighbors: Colocated={safe_format(relevant.get('colocated_neighbor_count'), '', 0)}, Non-colocated={safe_format(relevant.get('noncolocated_neighbor_count'), '', 0)}
• PCI pattern: Check for mod-30 collisions, PCI reuse, confusion between cells
• Abnormal path loss: {safe_format(relevant.get('abnormal_path_loss'))}
• Tilt config: Electronic={safe_format(relevant.get('serving_electronic_tilt_deg'), '°')}
• Not typical interference (C4) or coverage (C1) pattern → PCI-specific issue
**Root Cause:** C6 - PCI collision/confusion (mod-30 collision, PCI reuse causing cell ID ambiguity)""",

        "C7": f"""**Analysis:**
• Speed: max={safe_format(relevant.get('speed_max_kmh'), ' km/h')} (>40 threshold), mean={safe_format(relevant.get('speed_mean_kmh'), ' km/h')}
• Speed flag: {safe_format(relevant.get('speed_above_40_flag'))}, C7 hint: {safe_format(relevant.get('c7_speed_hint'))}
• Mobility impact: Fast+low TP ratio={safe_format(relevant.get('fast_low_tp_ratio'))}
• Speed-TP correlation: {safe_format(relevant.get('speed_tp_correlation'))}
• Performance degrades at high speed (not static coverage/interference)
• Likely cause: Doppler, tracking issues, HO delays at high mobility
**Root Cause:** C7 - High speed impact (>40 km/h mobility degradation)""",

        "C8": f"""**Analysis:**
• Resources: RB mean={safe_format(relevant.get('rb_mean'))}, RB min={safe_format(relevant.get('rb_min'), '', 0)}
• Efficiency: High RB+low TP ratio={safe_format(relevant.get('high_rb_low_tp_ratio_v2'))}
• TP drop with high RB: {safe_format(relevant.get('tp_drop_with_high_rb_ratio'))}
• RB-TP correlation: {safe_format(relevant.get('rb_tp_correlation'))} (weak/negative)
• TP efficiency min: {safe_format(relevant.get('throughput_efficiency_min'))}
• Pattern: High resource allocation but low throughput realization (not coverage/interference)
**Root Cause:** C8 - Resource congestion (high RB allocation, poor scheduling/backhaul)"""
    }

    return reasoning_templates.get(answer, "**Analysis:** Based on the features provided...")

In [64]:
import pandas as pd
import re
from typing import Dict, Any, List, Optional

# -------------------------
# A) Helpers
# -------------------------
def boxed(token: Optional[str]) -> str:
    token = "" if token is None else str(token).strip()
    return f"\\boxed{{{token}}}"

def build_user_prompt_rca(question: str, features_text: str) -> str:
    # Keep it compact: question + engineered features
    return f"{question}\n\n{features_text}".strip()

def build_user_prompt_general(question: str) -> str:
    # General MCQ: keep as-is
    return question.strip()

def build_assistant_general(answer_token: str) -> str:
    # For general tasks, do not force chain-of-thought. Just boxed answer.
    return boxed(answer_token)

def build_assistant_rca(answer_token: str, features_dict: Dict[str, Any]) -> str:
    # Convert token -> canonical label if token is like "P7" or "7" etc.
    # For your reasoning generator, it expects "C1..C8".
    # We can map token back to canonical cause using option-text mapping (from Step 2).
    # If you already stored canonical_cause, use that.
    canonical = None
    if isinstance(answer_token, str):
        # If it's already C1..C8 keep it
        if re.fullmatch(r"C[1-8]", answer_token.strip(), flags=re.IGNORECASE):
            canonical = answer_token.strip().upper()

    # Prefer canonical_cause if present in row; handled in apply below.
    if canonical is None:
        # Fallback: if token contains a digit 1..8 interpret as cause number
        m = re.search(r"([1-8])", str(answer_token))
        canonical = f"C{m.group(1)}" if m else None

    if not canonical:
        return boxed(answer_token)

    reasoning = generate_reasoning_for_class(canonical, features_dict)
    return f"{reasoning}\n\n**Final Answer:** {boxed(answer_token)}"


# -------------------------
# B) System prompt (temporary; we refine in Step 5)
# -------------------------
SYSTEM_PROMPT_V1 = """You solve two types of questions:

Type A (5G RCA): Use the provided drive-test + engineering info and the computed RCA feature summary to infer the most likely root cause.
Type B (General): Answer the multiple-choice question using reasoning.

Output rule (all types): End with a final answer in \\boxed{...} matching the option style in the question (e.g., 3, P7, M4, A)."""

# -------------------------
# C) Build message records
# -------------------------
def build_records_for_sft(df: pd.DataFrame, name: str, include_assistant: bool) -> List[Dict[str, Any]]:
    records = []

    for r in df.to_dict("records"):
        q = str(r.get("question", "")).strip()
        qtype = r.get("qtype", None)

        # If qtype missing, infer quickly
        if not qtype:
            qtype = "A_RCA" if ("features_text" in r and r.get("features_text")) else "B_GENERAL"

        if qtype == "A_RCA":
            user = build_user_prompt_rca(q, str(r.get("features_text", "")).strip())
        else:
            user = build_user_prompt_general(q)

        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT_V1},
            {"role": "user", "content": user},
        ]

        if include_assistant:
            ans = r.get("answer")
            if ans is None or (isinstance(ans, float) and pd.isna(ans)):
                # train/val should have answers; skip if missing
                continue

            ans = str(ans).strip()

            if qtype == "A_RCA":
                feats = r.get("features_dict") or {}
                # If canonical_cause present, prefer it for reasoning generator
                canonical = r.get("canonical_cause")
                if canonical:
                    assistant = f"{generate_reasoning_for_class(canonical, feats)}\n\n**Final Answer:** {boxed(ans)}"
                else:
                    assistant = build_assistant_rca(ans, feats)
            else:
                assistant = build_assistant_general(ans)

            msgs.append({"role": "assistant", "content": assistant})

        records.append({
            "id": r.get("ID", r.get("id")),
            "qtype": qtype,
            "messages": msgs
        })

    print(f"✓ {name}: built {len(records)} records (include_assistant={include_assistant})")
    return records


# Train/Val: include assistant
train_records = build_records_for_sft(train_df_aug, "train_df_aug", include_assistant=True)
val_records   = build_records_for_sft(val_df_aug,   "val_df_aug",   include_assistant=True)

# Test: prompts only (no assistant)
test_prompts  = build_records_for_sft(test_df,      "test_df",      include_assistant=False)


✓ train_df_aug: built 3730 records (include_assistant=True)
✓ val_df_aug: built 1344 records (include_assistant=True)
✓ test_df: built 863 records (include_assistant=False)
